In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import pandas as pd
bikes = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes.csv')
bikes.head()
# import pandas as pl
# bikes = pl.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes.csv')

,dteday,hr,casual,registered,temp_c,feels_like_c,hum,windspeed,weathersit,season,holiday,workingday
0,1/1/2011,0.0,3,13,3.0,3.0,0.7957,0.8,1,1,0,0
1,1/1/2011,1.0,8,30,1.7,1.7,0.8272,0.8,1,1,0,0
2,1/1/2011,2.0,5,26,1.9,1.9,0.8157,1.1,1,1,0,0
3,1/1/2011,3.0,3,9,2.5,2.5,0.7831,0.8,1,1,0,0
4,1/1/2011,4.0,0,1,2.0,2.0,0.8075,1.1,1,1,0,0


In [2]:
bikes['total'] = bikes['casual'] + bikes['registered']
bikes.head()

,dteday,hr,casual,registered,temp_c,feels_like_c,hum,windspeed,weathersit,season,holiday,workingday,total
0,1/1/2011,0.0,3,13,3.0,3.0,0.7957,0.8,1,1,0,0,16
1,1/1/2011,1.0,8,30,1.7,1.7,0.8272,0.8,1,1,0,0,38
2,1/1/2011,2.0,5,26,1.9,1.9,0.8157,1.1,1,1,0,0,31
3,1/1/2011,3.0,3,9,2.5,2.5,0.7831,0.8,1,1,0,0,12
4,1/1/2011,4.0,0,1,2.0,2.0,0.8075,1.1,1,1,0,0,1


In [3]:
bikes['dteday'] = pd.to_datetime(bikes['dteday'])
# Extract day of year and add hour as a fraction
bikes['day_hour'] = bikes['dteday'].dt.dayofyear + (bikes['hr'] / 24)

display(bikes[['dteday', 'hr', 'day_hour']].head())

,dteday,hr,day_hour
0,2011-01-01,0.0,1.000000
1,2011-01-01,1.0,1.041667
2,2011-01-01,2.0,1.083333
3,2011-01-01,3.0,1.125000
4,2011-01-01,4.0,1.166667


In [4]:
X = bikes.drop(columns=['casual', 'registered', 'dteday', 'hr', 'total'])
y = bikes['total']

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [6]:
y_train

,total
95962,138
34637,191
5954,1
109932,34
103772,1029
...,...
76820,100
110268,51
103694,18
860,67


In [7]:
from sklearn.preprocessing import MinMaxScaler

# fit scaler on training data
norm = MinMaxScaler().fit(X_train)

# transform training data
X_train = norm.transform(X_train)

# transform testing dataabs
X_test = norm.transform(X_test)



GRID SEARCH

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam, RMSprop
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import GridSearchCV

# Define the function to create the model
def create_model(optimizer='adam', dropout_rate=0.5, activation='relu'):
    model = Sequential()
    model.add(Input(shape=(len(X_train[0]),)))  # Use Input layer instead of input_dim
    model.add(Dense(128, activation=activation))
    model.add(Dropout(dropout_rate))
    model.add(Dense(256, activation=activation))
    model.add(Dense(64, activation='leaky_relu'))
    model.add(Dense(1, activation='relu'))
    model.compile(loss='mse', optimizer=optimizer)
    return model

# Wrap the model with KerasRegressor for compatibility with GridSearchCV
model = KerasRegressor(model=create_model, verbose=1)

# Define the grid of hyperparameters
param_grid = {
    # 'model__optimizer': ['adam', 'rmsprop'],
    'model__dropout_rate': [0.3, 0.5, 0.7],
    'model__activation': ['relu', 'tanh'],
    'batch_size': [16, 32, 64],
    'epochs': [10]  # You can reduce the number of epochs to speed up testing
}

# Set up GridSearchCV
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring='neg_mean_squared_error', cv=3)

# Execute the grid search
grid_result = grid.fit(X_train, y_train)

# Output the best parameters and score
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))


AttributeError: 'super' object has no attribute '__sklearn_tags__'